# CoinMate ML Model Training
업비트 전 종목 200일치 OHLCV로 XGBoost 모델 학습

**목표**: EC2보다 높은 정확도의 모델을 학습하여 `.pkl` 파일로 내보내기

In [ ]:
# 1. 필수 패키지 설치
!pip install -q pyupbit xgboost scikit-learn joblib lightgbm optuna

In [ ]:
# 2. 데이터 수집 (전 종목 200일치)
import pyupbit
import pandas as pd
import numpy as np
import time
from datetime import datetime

print('업비트 KRW 마켓 전 종목 OHLCV 수집 시작...')
tickers = pyupbit.get_tickers(fiat='KRW')
print(f'총 {len(tickers)}개 종목')

all_ohlcv = {}
failed = []

for i, ticker in enumerate(tickers):
    try:
        df = pyupbit.get_ohlcv(ticker, interval='day', count=200)
        if df is not None and len(df) >= 60:
            all_ohlcv[ticker] = df
        time.sleep(0.1)  # API 속도 제한
    except Exception as e:
        failed.append(ticker)
    
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(tickers)} 완료 ({len(all_ohlcv)}개 수집)')

print(f'\n수집 완료: {len(all_ohlcv)}개 종목 (실패: {len(failed)}개)')

In [ ]:
# 3. 피처 엔지니어링 (29개 피처)

def build_features(df):
    """OHLCV DataFrame에서 ML 피처 생성"""
    if df is None or len(df) < 50:
        return pd.DataFrame()

    feat = pd.DataFrame(index=df.index)
    close = df['close']
    high = df['high']
    low = df['low']
    volume = df['volume']
    open_p = df['open']

    # 가격 수익률
    feat['return_1d'] = close.pct_change(1) * 100
    feat['return_3d'] = close.pct_change(3) * 100
    feat['return_5d'] = close.pct_change(5) * 100
    feat['return_10d'] = close.pct_change(10) * 100

    # 이동평균 비율
    ma5 = close.rolling(5).mean()
    ma10 = close.rolling(10).mean()
    ma20 = close.rolling(20).mean()
    feat['ma5_ratio'] = (close / ma5 - 1) * 100
    feat['ma10_ratio'] = (close / ma10 - 1) * 100
    feat['ma20_ratio'] = (close / ma20 - 1) * 100
    feat['ma5_ma20_cross'] = (ma5 / ma20 - 1) * 100

    # RSI
    delta = close.diff()
    gain = delta.where(delta > 0, 0).ewm(alpha=1/14, min_periods=14).mean()
    loss = -delta.where(delta < 0, 0).ewm(alpha=1/14, min_periods=14).mean()
    rs = gain / loss.replace(0, 0.0001)
    feat['rsi'] = 100 - (100 / (1 + rs))

    # MFI
    tp = (high + low + close) / 3
    mf = tp * volume
    pos_flow = pd.Series(0.0, index=df.index)
    neg_flow = pd.Series(0.0, index=df.index)
    tp_delta = tp.diff()
    pos_flow[tp_delta > 0] = mf[tp_delta > 0]
    neg_flow[tp_delta < 0] = mf[tp_delta < 0]
    pos_sum = pos_flow.rolling(14).sum()
    neg_sum = neg_flow.rolling(14).sum().replace(0, 0.0001)
    feat['mfi'] = 100 - (100 / (1 + pos_sum / neg_sum))

    # MACD
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    feat['macd_diff'] = macd_line - signal_line
    feat['macd_ratio'] = feat['macd_diff'] / close * 100

    # ADX
    up_move = high.diff()
    down_move = -low.diff()
    pdm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    mdm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    pdm_s = pd.Series(pdm, index=df.index).ewm(alpha=1/14, min_periods=14).mean()
    mdm_s = pd.Series(mdm, index=df.index).ewm(alpha=1/14, min_periods=14).mean()
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    tr_smooth = tr.ewm(alpha=1/14, min_periods=14).mean().replace(0, 0.0001)
    pdi = 100 * pdm_s / tr_smooth
    mdi = 100 * mdm_s / tr_smooth
    dx = (abs(pdi - mdi) / (pdi + mdi).replace(0, 0.0001)) * 100
    feat['adx'] = dx.ewm(alpha=1/14, min_periods=14).mean()
    feat['di_diff'] = pdi - mdi

    # 볼린저 밴드
    bb_ma = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    feat['bb_position'] = (close - (bb_ma - 2 * bb_std)) / (4 * bb_std.replace(0, 0.0001))

    # 거래량
    vol_ma20 = volume.rolling(20).mean().replace(0, 1)
    feat['vol_ratio'] = volume / vol_ma20
    feat['vol_change'] = volume.pct_change(1) * 100

    # ATR
    atr = tr.rolling(14).mean()
    feat['atr_ratio'] = atr / close * 100

    # 캔들 패턴
    body = abs(close - open_p)
    total_range = (high - low).replace(0, 0.0001)
    feat['body_ratio'] = body / total_range
    feat['upper_wick'] = (high - pd.concat([close, open_p], axis=1).max(axis=1)) / total_range
    feat['is_bullish'] = (close > open_p).astype(int)

    # 변동성
    feat['volatility_5d'] = close.pct_change().rolling(5).std() * 100
    feat['volatility_10d'] = close.pct_change().rolling(10).std() * 100

    # 시간 패턴
    if hasattr(df.index, 'hour'):
        feat['hour'] = df.index.hour
    else:
        feat['hour'] = 12

    # 모멘텀 가속도
    feat['momentum_accel'] = feat['return_1d'].diff()

    # 거래량-가격 상관
    feat['vol_price_corr'] = close.rolling(5).corr(volume)

    # 고저 범위
    feat['hl_range_pct'] = ((high - low) / close) * 100

    # RSI 변화율
    feat['rsi_change'] = feat['rsi'].diff()

    # 연속 양봉/음봉
    is_up = (close > open_p).astype(int)
    groups = (is_up != is_up.shift()).cumsum()
    feat['consecutive_candles'] = is_up.groupby(groups).cumsum() - (1 - is_up).groupby(groups).cumsum()

    feat = feat.replace([np.inf, -np.inf], np.nan)
    return feat

print('피처 엔지니어링 함수 정의 완료 (29개 피처)')

In [ ]:
# 4. 학습 데이터 구축

all_X = []
all_y = []
coin_count = 0

for ticker, df in all_ohlcv.items():
    features = build_features(df)
    if features.empty:
        continue

    # 정답: 다음날 종가가 오늘보다 높으면 1
    y = (df['close'].shift(-1) > df['close']).astype(int)

    valid_mask = features.notna().all(axis=1) & y.notna()
    features = features[valid_mask]
    y = y[valid_mask]

    if len(features) < 30:
        continue

    all_X.append(features)
    all_y.append(y)
    coin_count += 1

X = pd.concat(all_X, ignore_index=True)
y = pd.concat(all_y, ignore_index=True)

print(f'학습 데이터: {coin_count}개 종목, {len(X):,}행 x {len(X.columns)}열')
print(f'클래스 비율: 상승 {y.mean():.1%} / 하락 {1-y.mean():.1%}')
print(f'피처 목록: {list(X.columns)}')

In [ ]:
# 5. Train/Test 분할 (시계열 — 마지막 20% 테스트)
from sklearn.model_selection import TimeSeriesSplit

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {len(X_train):,}행 / Test: {len(X_test):,}행')

In [ ]:
# 6. Optuna 하이퍼파라미터 자동 튜닝
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 20),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 5.0),
        'random_state': 42,
        'eval_metric': 'logloss',
        'verbosity': 0,
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    pred = model.predict(X_test)
    return accuracy_score(y_test, pred)

print('Optuna 튜닝 시작 (50 trials)...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n최적 정확도: {study.best_value:.4f} ({study.best_value*100:.1f}%)')
print(f'최적 파라미터: {study.best_params}')

In [ ]:
# 7. 최적 파라미터로 최종 모델 학습
from sklearn.metrics import classification_report, confusion_matrix

best_params = study.best_params
best_params['random_state'] = 42
best_params['eval_metric'] = 'logloss'
best_params['verbosity'] = 0

final_model = XGBClassifier(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

train_acc = final_model.score(X_train, y_train) * 100
test_acc = final_model.score(X_test, y_test) * 100

print(f'Train 정확도: {train_acc:.1f}%')
print(f'Test 정확도:  {test_acc:.1f}%')
print(f'\n분류 리포트:')
print(classification_report(y_test, final_model.predict(X_test), target_names=['하락', '상승']))
print(f'혼동 행렬:')
print(confusion_matrix(y_test, final_model.predict(X_test)))

In [ ]:
# 8. LightGBM 비교 학습
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=best_params.get('n_estimators', 300),
    max_depth=best_params.get('max_depth', 4),
    learning_rate=best_params.get('learning_rate', 0.03),
    subsample=best_params.get('subsample', 0.75),
    colsample_bytree=best_params.get('colsample_bytree', 0.7),
    min_child_weight=best_params.get('min_child_weight', 10),
    reg_alpha=best_params.get('reg_alpha', 0.3),
    reg_lambda=best_params.get('reg_lambda', 2.0),
    random_state=42,
    verbosity=-1,
)
lgbm.fit(X_train, y_train)

lgbm_train = lgbm.score(X_train, y_train) * 100
lgbm_test = lgbm.score(X_test, y_test) * 100
print(f'LightGBM Train: {lgbm_train:.1f}% / Test: {lgbm_test:.1f}%')

# 더 좋은 모델 선택
if lgbm_test > test_acc:
    print(f'\n>>> LightGBM이 더 좋음! ({lgbm_test:.1f}% > {test_acc:.1f}%)')
    best_model = lgbm
    best_model_name = 'LightGBM'
    best_test_acc = lgbm_test
else:
    print(f'\n>>> XGBoost가 더 좋거나 같음! ({test_acc:.1f}% >= {lgbm_test:.1f}%)')
    best_model = final_model
    best_model_name = 'XGBoost'
    best_test_acc = test_acc

In [ ]:
# 9. 피처 중요도 시각화
import matplotlib.pyplot as plt

if best_model_name == 'XGBoost':
    importances = final_model.feature_importances_
else:
    importances = lgbm.feature_importances_

feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
feat_imp.plot(kind='barh', color='steelblue')
plt.title(f'Feature Importance ({best_model_name})')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# 10. 모델 저장 (.pkl) — EC2 호환 형식
import joblib
from datetime import timezone, timedelta

KST = timezone(timedelta(hours=9))
today = datetime.now(KST).strftime('%Y-%m-%d')

model_data = {
    'model': best_model,
    'feature_names': list(X.columns),
    'score': best_test_acc,
    'train_date': today,
    'model_name': best_model_name,
    'n_coins': coin_count,
    'n_samples': len(X),
    'best_params': best_params if best_model_name == 'XGBoost' else None,
}

filename = f'xgb_model.pkl'
joblib.dump(model_data, filename)
print(f'모델 저장 완료: {filename}')
print(f'  모델: {best_model_name}')
print(f'  정확도: {best_test_acc:.1f}%')
print(f'  피처: {len(X.columns)}개')
print(f'  학습일: {today}')
print(f'  학습 데이터: {coin_count}개 종목 / {len(X):,}행')

In [ ]:
# 11. 파일 다운로드 (Colab → 로컬)
from google.colab import files
files.download('xgb_model.pkl')
print('\n다운로드 시작!')
print('다운받은 파일을 EC2에 업로드하세요:')
print('scp -i coinmate.pem xgb_model.pkl ec2-user@15.134.82.85:~/CoinMate-Backend/models/')